# Running Your First Eval with FutureAGI

Score LLM outputs for hallucination, toxicity, and custom quality criteria — all in one notebook.

By the end of this notebook you will have scored LLM responses using three approaches:
1. **Local metrics** — fast string/similarity checks, zero API keys
2. **FutureAGI Turing models** — purpose-built evaluation models for quality, safety, and semantics
3. **LLM-as-Judge** — custom evaluation criteria using any LLM

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

In [ ]:
%pip install 'ai-evaluation[nli]' --quiet

The `[nli]` extra installs the local NLI model used by `faithfulness` and `contradiction_detection` (Step 2). Without it, these metrics fall back to a less accurate word-overlap heuristic.

---
## Step 1: Run a local metric (no API key)

Local metrics run entirely on your machine. No network call, no API key, instant.

In [ ]:
from fi.evals import evaluate

# Check if a response contains expected content
result = evaluate("contains", output="Your order has shipped!", keyword="shipped")

print(f"Score:  {result.score}")
print(f"Passed: {result.passed}")
print(f"Reason: {result.reason}")

In [ ]:
from fi.evals import evaluate

# Exact match
print("equals:", evaluate("equals", output="Paris", expected_output="Paris").passed)

# Format check
print("is_json:", evaluate("is_json", output='{"status": "ok"}').passed)

# Length check
print("length:", evaluate("length_less_than", output="Short reply.", max_length=100).passed)

# String similarity
result = evaluate("levenshtein_similarity", output="colour", expected_output="color")
print(f"levenshtein: score={result.score}")

---
## Step 2: Detect contradictions with local NLI

Uses a local NLI (natural language inference) model — no API key required.

In [ ]:
from fi.evals import evaluate

# Supported response
result = evaluate(
    "contradiction_detection",
    output="The Eiffel Tower is located in Paris, France.",
    context="The Eiffel Tower is a wrought-iron lattice tower located in Paris.",
)
print(f"Score:  {result.score:.2f}")
print(f"Passed: {result.passed}\n")

# Contradictory response
result = evaluate(
    "contradiction_detection",
    output="The Eiffel Tower is located in London, England.",
    context="The Eiffel Tower is a wrought-iron lattice tower located in Paris.",
)
print(f"Score:  {result.score:.2f}")
print(f"Passed: {result.passed}")
print(f"Reason: {result.reason}")

---
## Step 3: Score with FutureAGI's Turing evaluation models

For quality, tone, safety, and semantic evaluations, use FutureAGI's Turing models. These are purpose-built evaluation models, not general-purpose LLMs.

Available models:
- `turing_flash` — lowest latency, best for high-volume pipelines
- `turing_small` — balanced speed and accuracy (recommended default)
- `turing_large` — highest accuracy, supports audio and PDF inputs

In [ ]:
import os

# Set your FutureAGI API keys
os.environ["FI_API_KEY"] = "your-api-key"        # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"  # Replace with your key

In [ ]:
from fi.evals import evaluate

# Toxicity check — safe response
result = evaluate(
    "toxicity",
    output="You're amazing, keep it up!",
    model="turing_small",
)
print(f"Toxicity score: {result.score}")
print(f"Passed: {result.passed}\n")

# Toxicity check — problematic response
result = evaluate(
    "toxicity",
    output="I hate you and everything you stand for.",
    model="turing_small",
)
print(f"Score:  {result.score}")
print(f"Reason: {result.reason}")

Explore all 72+ [built-in eval metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview): `tone`, `context_adherence`, `completeness`, `groundedness`, `data_privacy`, `bias_detection`, `instruction_adherence`, and more.

---
## Step 4: Run multiple metrics at once

Pass a list of metric names to run several evals on the same output in one call. Returns a `BatchResult` you can iterate.

In [ ]:
from fi.evals import evaluate

results = evaluate(
    ["toxicity", "groundedness"],
    output="The Eiffel Tower is located in Paris, France.",
    context="The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris.",
    input="Where is the Eiffel Tower?",
    model="turing_small",
)

for result in results:
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<20} score={result.score}  {status}")
    print(f"  Reason: {result.reason}\n")

---
## Step 5: Write your own evaluation criteria (LLM-as-Judge)

When no built-in metric fits, describe your quality bar in plain English and use any LLM as the judge.

Any [LiteLLM model string](https://docs.litellm.ai/docs/providers) works: `gpt-4o`, `claude-sonnet-4-20250514`, `gemini/gemini-2.5-flash`, `ollama/llama3.2:3b`.

In [ ]:
import os

# Set your LLM provider key (any LiteLLM-supported provider)
os.environ["GOOGLE_API_KEY"] = "your-google-api-key"  # or OPENAI_API_KEY, ANTHROPIC_API_KEY

In [ ]:
from fi.evals import evaluate

result = evaluate(
    prompt="""You are evaluating a customer support response.

    Score 1.0 if the response:
    - Acknowledges the customer's issue clearly
    - Offers a concrete next step or resolution
    - Stays professional and empathetic

    Score 0.5 if it's polite but vague (no clear next step).
    Score 0.0 if it's dismissive, rude, or unhelpful.""",
    output="I understand your frustration with the delayed shipment. I've escalated this to our logistics team and you'll receive a status update within 2 hours.",
    input="My order is 3 weeks late and nobody is responding to my emails.",
    engine="llm",
    model="gemini/gemini-2.5-flash",
)

print(f"Score:  {result.score}")
print(f"Reason: {result.reason}")

---
## What you built

- Ran local string and similarity metrics in under 1ms with zero credentials
- Detected contradictions using a local NLI model
- Scored content quality and safety with FutureAGI's Turing evaluation models
- Ran multiple metrics on one output with `evaluate([...])` returning a `BatchResult`
- Defined custom evaluation criteria in plain English using LLM-as-Judge

### Next steps

- [All Built-in Metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) — 72+ metrics for safety, quality, and grounding
- [Custom Eval Metrics](https://docs.futureagi.com/cookbook/quickstart/custom-eval-metrics) — create reusable evaluation templates
- [Hallucination Detection](https://docs.futureagi.com/cookbook/quickstart/hallucination-detection) — faithfulness, groundedness, and context adherence
- [Eval in CI/CD](https://docs.futureagi.com/cookbook/quickstart/cicd-eval-pipeline) — block bad prompts with GitHub Actions